# Part 1 — Gemma 3 4B from Kaggle
Technical setup only. No scientific pilot or full evaluation. Run cells in order. Select a free GPU in Runtime → Change runtime type. Do not purchase compute. The R2 precision policy is BF16; unsupported hardware stops rather than silently using FP16 or quantization.

Current validation: CPU tests pass locally. Colab diagnostic execution was attempted but failed with a runtime network error. This notebook is not claimed GPU-validated.

In [ ]:
import sys, json, shutil, importlib.metadata as metadata
import torch
print('Python:',sys.version)
print('PyTorch:',torch.__version__, 'CUDA:',torch.version.cuda)
print('Free disk bytes:',shutil.disk_usage('/content').free)
print('CUDA available:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
    print('GPU bytes:',torch.cuda.get_device_properties(0).total_memory)
    print('BF16 supported:',torch.cuda.is_bf16_supported())

## Acquire the project
Use the published commit once GitHub access is repaired. Until then, upload and extract the supplied project ZIP through Colab Files. Set PROJECT to the extracted research_project folder. Never upload credentials with the project.

In [ ]:
from pathlib import Path
import subprocess, os
PROJECT=Path('/content/contaxt-rot')
if not PROJECT.exists():
    subprocess.run(['git','clone','https://github.com/shehzan707/contaxt-rot.git',str(PROJECT)],check=True)
assert (PROJECT/'src/kaggle_model.py').exists(), 'Repository import is blocked. Extract the supplied ZIP and set PROJECT to its research_project directory.'
os.chdir(PROJECT)
print('Project:',PROJECT)

## Dependencies
Candidate ranges are not a GPU-tested lock. Retain the runtime PyTorch/CUDA pair. Restart the runtime if package installation requests it, then rerun diagnostics and acquisition.

In [ ]:
%pip install -r requirements-colab.txt
!python -m unittest discover -s tests -v
!python scripts/diagnostics.py
!python -m pip freeze > reports/colab_packages.txt

## Kaggle access — user action
Open https://www.kaggle.com/models/google/gemma-3/transformers/gemma-3-4b-it and review Request Access terms yourself. If acceptable, complete consent. Create a token yourself at https://www.kaggle.com/settings/api and save it in Colab Secrets as KAGGLE_API_TOKEN; enable access for this notebook. Do not paste it into a cell or share it. Browser sign-in alone does not authenticate this runtime.

In [ ]:
from google.colab import userdata
import os
try:
    os.environ['KAGGLE_API_TOKEN']=userdata.get('KAGGLE_API_TOKEN')
except Exception:
    raise RuntimeError('Add KAGGLE_API_TOKEN in Colab Secrets and grant notebook access.') from None
print('Secret loaded without displaying its value.')

## Version-pinned checkpoint and processor
Official Google Transformers variation, version 1. Downloading requires your prior licence consent. This step hashes every checkpoint file and can take several minutes. No Hugging Face account is required for Gemma.

In [ ]:
import json
from src.kaggle_model import checkpoint
config=json.loads(Path('configs/protocol.json').read_text())
spec=config['models'][0]
model_path,model_manifest=checkpoint(spec)
Path('reports/checkpoint_manifest.json').write_text(json.dumps(model_manifest,indent=2))
from transformers import AutoProcessor
processor=AutoProcessor.from_pretrained(model_path,local_files_only=True)
assert processor.tokenizer.chat_template
print('Checkpoint:',spec['handle'])
print('Native token check:',len(processor.tokenizer.encode('Technical test',add_special_tokens=False)))

## Bounded GPU smoke checks
Run one short non-research prompt and 4K/16K synthetic memory probes, at most 128 generated tokens each. BF16 unsupported hardware stops before loading. No quantization fallback. A failed probe is a technical outcome, not a scientific result.

In [ ]:
import subprocess
subprocess.run([sys.executable,'-m','scripts.smoke'],check=True)
rows=[json.loads(x) for x in Path('results/technical_smoke.jsonl').read_text().splitlines()]
assert all(r['non_research'] for r in rows)
for r in rows:
    print({k:r.get(k) for k in ['target_context_tokens','native_input_tokens','output_tokens','generation_seconds','peak_allocated_bytes','status','error']})

## Real benchmark preparation
These commands download public benchmark data and allocate questions only. They do not call Gemma. The CMU host returned 503 during setup; see the documented public Parquet fallback in PART1_SETUP_GUIDE.md. Human-review queues remain pending until reviewed.

In [ ]:
!python scripts/download_hotpot.py
!python -m src.prepare data/hotpot_dev_distractor_v1.json data/prepared
!python -m src.build_contexts queue data/prepared --splits pilot

## Save logs and release memory
Download the log archive before ending the runtime. It excludes secrets and weights. Use Runtime → Disconnect and delete runtime when finished. Do not run Part 2 until every readiness blocker is resolved.

In [ ]:
import zipfile, gc
with zipfile.ZipFile('/content/Part1_Technical_Logs.zip','w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['reports','results']:
        for f in Path(folder).glob('*'):
            if f.is_file() and f.suffix in ['.json','.jsonl','.txt','.md']:
                z.write(f,str(f))
os.environ.pop('KAGGLE_API_TOKEN',None)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
from google.colab import files
files.download('/content/Part1_Technical_Logs.zip')